# Local Search Intelligence — Full Panel findings (`FULLPANEL-202609-G13E` + `AIO-20260918`)

Renders the headline research findings from the analysis layer
(`supabase/migrations/027_analysis_layer_v0_1.sql`, schema `analysis`) over the
first full Maps + Organic panel on the efficient `GEOGRID13E_V1` grid, plus the
graduated AIO wave.

**Outcome-only, no composite score, missing ≠ zero** (see
`docs/design/analysis-layer-v0_1.md`). No paid call is made by this notebook.

Each section names the canonical `analysis.*` view it corresponds to. The heaviest
full-panel rollups are run here as the same set-based logic with
`statement_timeout = 600s` on a direct connection (the 60 s pooler/MCP window is too
small for a 1.4 M-row scan).

> Real numbers computed on production 2026-09-18/19 are quoted inline so this
> notebook reads as a findings record even before you re-run it.


In [ ]:
import os, textwrap
import pandas as pd
import psycopg
import matplotlib
import matplotlib.pyplot as plt

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

DSN = os.environ.get("SUPABASE_DB_URL")
assert DSN, ("Set SUPABASE_DB_URL (production session pooler) in the environment. "
             "Secrets live only in Railway/Supabase — never in this repo.")

WAVE_MAPORG = "FULLPANEL-202609-G13E"
WAVE_AIO    = "AIO-20260918"

# Colorblind-safe qualitative palette (Okabe-Ito subset).
PAL = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9"]

def q(sql, params=None):
    """Run one query on a fresh connection with a generous statement timeout
    (the 60 s pooler/MCP window is too small for a full-panel scan) and return a
    DataFrame."""
    with psycopg.connect(DSN) as c:
        c.execute("set statement_timeout = '600s'")
        cur = c.execute(sql, params or {})
        cols = [d.name for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)

print("connected:", DSN.split("@")[-1] if "@" in DSN else "(dsn set)")


## 0. Panel shape & coverage  → `analysis.coverage_summary`

The Full Panel is **595 eligible points × 25 industries × 4 queries × 2 surfaces =
119,000 executable observations**; 11,000 structural coordinates (10,000 water +
1,000 outside-country) are excluded and carry no observation (missing ≠ zero).

**Computed: 119,000 / 119,000 eligible observations returned — 100 % coverage.**


In [ ]:
cov = q('''
  select surface_code,
         count(*) as observations,
         count(distinct coordinate_id) as observed_points,
         round(avg(nr)::numeric, 3) as avg_results_per_point
  from (
    select s.surface_code, o.observation_id, j.coordinate_id,
      case s.surface_code
        when 'maps'    then (select count(*) from maps.result r
                               where r.observation_id=o.observation_id and r.provider_item_type='maps_search')
        else (select count(*) from organic.result r
                where r.observation_id=o.observation_id and r.result_type='organic')
      end as nr
    from ops.collection_wave w
    join ops.collection_job j on j.wave_id=w.wave_id
    join manifest.surface s on s.surface_id=j.surface_id
    join ops.observation o on o.job_id=j.job_id
    where w.wave_code=%(w)s
  ) t
  group by surface_code order by surface_code
''', {"w": WAVE_MAPORG})
cov


## 1. AIO prevalence at full-panel scale  → `analysis.aio_overview_prevalence`

Prevalence read straight from the organic SERP (`organic.result.result_type =
'ai_overview'`), so it covers all 59,500 organic observations — not just a dedicated
AIO wave. Absence of the block is a valid negative.

**Computed (full panel):**

| query class | prevalence |
|---|---|
| Q1 baseline_implicit_local ("… near me") | **4.3 %** |
| Q4 explicit_geography ("… in [CITY]") | 9.7 % |
| Q2 recommendation_quality ("best … near me") | 11.7 % |
| Q3 industry_specific_high_need | **13.7 %** |
| **overall** | **9.86 %** |

A standalone AI Overview is **rare (~1 in 10)** for near-me local intent and strongly
query-dependent — bare "near me" triggers it least; quality/high-need modifiers ~3×
more. This is the number to size the AIO-widen decision against.


In [ ]:
prev = q('''
  with oo as (
    select o.observation_id, t.treatment_code
    from ops.collection_wave w
    join ops.collection_job j on j.wave_id=w.wave_id
    join manifest.surface s on s.surface_id=j.surface_id and s.surface_code='organic'
    join ops.observation o on o.job_id=j.job_id
    join manifest.surface_treatment st on st.surface_treatment_id=j.surface_treatment_id
    join manifest.treatment t on t.treatment_id=st.treatment_id
    where w.wave_code=%(w)s
  ),
  aio as (select distinct r.observation_id
          from oo join organic.result r on r.observation_id=oo.observation_id
          where r.result_type='ai_overview')
  select oo.treatment_code, count(*) obs, count(a.observation_id) aio_present,
         round(count(a.observation_id)::numeric/count(*),4) prevalence
  from oo left join aio a on a.observation_id=oo.observation_id
  group by oo.treatment_code order by prevalence
''', {"w": WAVE_MAPORG})
LABEL = {"Q1":"Q1 near me","Q2":"Q2 best near me","Q3":"Q3 high-need","Q4":"Q4 in [CITY]"}
prev["label"] = prev["treatment_code"].map(LABEL).fillna(prev["treatment_code"])
overall = prev["aio_present"].sum() / prev["obs"].sum()

fig, ax = plt.subplots(figsize=(7,3.6))
ax.bar(prev["label"], prev["prevalence"]*100, color=PAL[0])
ax.axhline(overall*100, color=PAL[3], ls="--", lw=1, label=f"overall {overall*100:.1f}%")
ax.set_ylabel("AIO prevalence (%)"); ax.set_title("Standalone AI Overview prevalence by query class (full panel)")
ax.legend(); fig.tight_layout(); plt.show()
prev[["treatment_code","label","obs","aio_present","prevalence"]]


## 2. Distance-decay by ring  → `analysis.maps_center_retention`

For "… near me" (Q1), what fraction of the **center** point's Local-Pack businesses
still appear as you move out? (canonical grain = resolved `business_location`; here
computed via the place_id on the observed object, which is the same key).

**Computed (center-retention, Q1):**

| industry | 3 mi | 4 mi | 5 mi |
|---|---|---|---|
| Locksmith (IND010) | 22.4 % | 8.7 % | 3.6 % |
| Urgent care (IND019) | 18.5 % | 6.8 % | 2.7 % |
| Chinese restaurant (IND022) | 20.3 % | 7.3 % | 2.9 % |

Steep, consistent proximity decay: ~1/5 of the center pack remains at 3 mi, ~3 % at
5 mi — strong per-coordinate proximity signal that validates the 13-point grid's
spatial resolution. Set `INDUSTRIES = None` to run all 25 (needs the 600 s timeout).


In [ ]:
INDUSTRIES = ['IND010','IND019','IND022']   # None => all 25 industries (slow)
ind_filter = "" if INDUSTRIES is None else "and i.industry_code = any(%(inds)s)"
params = {"w": WAVE_MAPORG}
if INDUSTRIES is not None: params["inds"] = INDUSTRIES

decay = q(f'''
  with m as (
    select i.industry_code, j.market_id, gp.distance_miles as ring,
           oo.raw_external_ids->>'place_id' as pid
    from ops.collection_wave w
    join manifest.industry i on true {ind_filter}
    join manifest.surface s on s.surface_code='maps'
    join manifest.treatment t on t.industry_id=i.industry_id and t.treatment_code='Q1' and t.treatment_set_code='GOOGLE_QUERY_V1'
    join manifest.surface_treatment st on st.treatment_id=t.treatment_id and st.surface_id=s.surface_id
    join ops.collection_job j on j.wave_id=w.wave_id and j.surface_treatment_id=st.surface_treatment_id and j.industry_id=i.industry_id
    join ops.observation o on o.job_id=j.job_id
    join manifest.market_coordinate mc on mc.coordinate_id=j.coordinate_id
    join manifest.geometry_point gp on gp.geometry_point_id=mc.geometry_point_id
    join maps.result r on r.observation_id=o.observation_id and r.provider_item_type='maps_search'
    join core.observed_object oo on oo.observed_object_id=r.observed_object_id
    where w.wave_code=%(w)s and oo.raw_external_ids->>'place_id' is not null
  ),
  center as (select distinct industry_code, market_id, pid from m where ring=0),
  csize as (select industry_code, market_id, count(*) n from center group by 1,2),
  pp as (
    select m.industry_code, m.market_id, m.ring, m.observation_id,
           count(distinct m.pid) filter (where c.pid is not null) retained
    from m left join center c on c.industry_code=m.industry_code and c.market_id=m.market_id and c.pid=m.pid
    where m.ring>0 group by 1,2,3,4
  )
  select pp.industry_code, pp.ring::float as ring,
         round(avg(pp.retained::numeric/nullif(csize.n,0)),4) as center_retention
  from pp join csize on csize.industry_code=pp.industry_code and csize.market_id=pp.market_id
  group by pp.industry_code, pp.ring order by pp.industry_code, pp.ring
''', params)

fig, ax = plt.subplots(figsize=(7,3.8))
for i,(code_,grp) in enumerate(decay.groupby("industry_code")):
    ax.plot(grp["ring"], grp["center_retention"]*100, "o-", color=PAL[i%len(PAL)], label=code_)
ax.set_xlabel("distance from center (miles)"); ax.set_ylabel("center Local-Pack retained (%)")
ax.set_title('Distance-decay of the Local Pack ("… near me")'); ax.legend(); fig.tight_layout(); plt.show()
decay.pivot(index="industry_code", columns="ring", values="center_retention")


## 3. Maps ↔ Organic overlap  → `analysis.maps_organic_overlap`

At the same point + query, how many of the Local Pack's **business-website domains**
also rank in the **organic** results?

**Computed (Locksmith Q1, 568 cells): only ~16 % overlap** — Local Pack averages
~6.1 website domains/cell, ~0.95 of which also rank organically. Maps (proximate GBP
businesses) and Organic (web publishers) reward largely **different** players.


In [ ]:
ov = q('''
  with base as (
    select j.market_id, j.coordinate_id, s.surface_code, o.observation_id
    from ops.collection_wave w
    join manifest.industry i on i.industry_code=%(ind)s
    join manifest.treatment t on t.industry_id=i.industry_id and t.treatment_code='Q1' and t.treatment_set_code='GOOGLE_QUERY_V1'
    join manifest.surface_treatment st on st.treatment_id=t.treatment_id
    join manifest.surface s on s.surface_id=st.surface_id and s.surface_code in ('maps','organic')
    join ops.collection_job j on j.wave_id=w.wave_id and j.surface_treatment_id=st.surface_treatment_id
    join ops.observation o on o.job_id=j.job_id
    where w.wave_code=%(w)s
  ),
  maps_dom as (
    select distinct b.market_id, b.coordinate_id, analysis.norm_domain(r.url_raw) d
    from base b join maps.result r on r.observation_id=b.observation_id and r.provider_item_type='maps_search'
    where b.surface_code='maps' and analysis.norm_domain(r.url_raw) is not null
  ),
  org_dom as (
    select distinct b.market_id, b.coordinate_id, analysis.norm_domain(coalesce(nullif(r.domain_raw,''),r.url_raw)) d
    from base b join organic.result r on r.observation_id=b.observation_id and r.result_type='organic'
    where b.surface_code='organic' and analysis.norm_domain(coalesce(nullif(r.domain_raw,''),r.url_raw)) is not null
  ),
  per_cell as (
    select md.market_id, md.coordinate_id, count(distinct md.d) maps_domains,
           count(distinct md.d) filter (where od.d is not null) overlap
    from maps_dom md
    left join org_dom od on od.market_id=md.market_id and od.coordinate_id=md.coordinate_id and od.d=md.d
    group by md.market_id, md.coordinate_id
  )
  select count(*) cells, round(avg(maps_domains),2) avg_maps_domains,
         round(avg(overlap),2) avg_overlap,
         round(avg(overlap::numeric/nullif(maps_domains,0)),4) avg_share_localpack_also_organic
  from per_cell
''', {"w": WAVE_MAPORG, "ind": "IND010"})
ov


## 4. Entity dominance  → `analysis.maps_entity_dominance` / `analysis.organic_domain_dominance`

Who blankets the grid? `coverage_share` = distinct eligible points where the
entity/domain appears ÷ the market's eligible points. Shown for one cell (Locksmith ×
Vancouver WA) to stay light; drop the market filter (with the 600 s timeout) for the
panel-wide leaderboard.


In [ ]:
topbiz = q('''
  select business_name, appearances, distinct_points, coverage_share, best_rank, avg_rank
  from analysis.maps_entity_dominance
  where wave_code=%(w)s and industry_code='IND010' and market_code='MKT008'
  order by coverage_share desc, appearances desc limit 10
''', {"w": WAVE_MAPORG})
topdom = q('''
  select domain, appearances, distinct_points, coverage_share, best_rank, avg_rank
  from analysis.organic_domain_dominance
  where wave_code=%(w)s and industry_code='IND010' and market_code='MKT008'
  order by coverage_share desc, appearances desc limit 10
''', {"w": WAVE_MAPORG})
print("Top Maps businesses (Locksmith × Vancouver WA):"); display(topbiz)
print("Top Organic domains (Locksmith × Vancouver WA):"); display(topdom)


## 5. AIO ↔ organic source overlap  → `analysis.aio_organic_source_overlap`

For the graduated AIO wave: when a standalone AIO fires and cites **web sources**
(reference cards), are those sources already ranking organically?

**Computed:** 5 of 30 obs triggered a standalone AIO (~17 %); of the 2 that cited web
sources, **100 % of AIO web sources (5/5) also rank in the co-returned organic
results**. Most triggered AIOs cite GBP-carousel/SearchViewer **businesses**, not web
sources — so AIO's *independent* signal is GBP embedding, while its web-citation layer
is redundant with organic. (small n → directional.)


In [ ]:
aio = q('''
  select treatment_code, aio_triggered, aio_presentation_form,
         aio_source_domains, organic_domains, overlap_domains, share_of_aio_sources_in_organic
  from analysis.aio_organic_source_overlap
  where wave_code=%(w)s and aio_triggered
  order by aio_source_domains desc
''', {"w": WAVE_AIO})
aio


## 6. Synthesis — implication for the AIO-widen go/no-go

- **Organic** and the **AIO sidebar** reward web publishers/brands — the AIO's web
  sources *are* the organic winners.
- **Maps** rewards *proximate GBP businesses*, largely disjoint from organic
  (~16 % overlap) and steeply proximity-decaying (~3 % of the center pack survives to
  5 mi).
- A **standalone AIO is rare (~10 %)** and query-dependent, and its web-citation layer
  is largely **redundant** with organic data already collected. Its only *independent*
  contribution is the **GBP carousel** (SearchViewer embedding).

**So a full 148,750-task AIO widen would mostly re-collect organic-redundant signal.**
The higher-value options are a **targeted** widen (high-AIO query classes — "best" /
high-need / in-[CITY]; and GBP-embedding capture) or **deferral** until enrichment
predictors exist. Confirm against the panel-wide generalizations above (set
`INDUSTRIES = None`) before deciding.
